# Build the reporting views

Generated from `fabric/05-gold.yaml` — do not edit by hand.

Runs **after every dimension and fact**, because these views select from tables that Spark creates through the warehouse connector. They cannot exist before the first load, which is why they are built here rather than by a deploy-time migration.

`CREATE OR ALTER`, so a re-run converges instead of failing.

In [ ]:
WAREHOUSE = 'wh_gold'
REPORTING_SCHEMA = 'bi'
PHYSICAL_SCHEMA = 'dbo'

VIEWS = ['vw_sales_summary', 'vw_product_performance', 'vw_channel_comparison']
STATEMENTS = ['CREATE OR ALTER VIEW [bi].[vw_sales_summary] AS\n'
 'SELECT\n'
 '  r.region_name,\n'
 '  s.salesperson_name,\n'
 '  p.product_name,\n'
 '  COUNT(DISTINCT f.sales_order_number) as order_count,\n'
 '  SUM(f.quantity) as total_quantity,\n'
 '  SUM(f.sales_revenue) as total_revenue,\n'
 '  SUM(f.sales_cost) as total_cost,\n'
 '  SUM(f.gross_profit) as total_profit\n'
 'FROM dbo.fct_sales f\n'
 'JOIN dbo.dim_region r ON f.region_sk = r.region_sk\n'
 'JOIN dbo.dim_salesperson s ON f.salesperson_sk = s.salesperson_sk\n'
 'JOIN dbo.dim_product p ON f.product_sk = p.product_sk\n'
 'GROUP BY r.region_name, s.salesperson_name, p.product_name',
 'CREATE OR ALTER VIEW [bi].[vw_product_performance] AS\n'
 'SELECT\n'
 '  p.product_name,\n'
 '  p.category,\n'
 '  p.subcategory,\n'
 '  COUNT(DISTINCT f.sales_order_number) as order_count,\n'
 '  SUM(f.quantity) as units_sold,\n'
 '  SUM(f.sales_revenue) as total_revenue,\n'
 '  SUM(f.sales_cost) as total_cost,\n'
 '  SUM(f.gross_profit) as total_profit\n'
 'FROM dbo.fct_sales f\n'
 'JOIN dbo.dim_product p ON f.product_sk = p.product_sk\n'
 'GROUP BY p.product_name, p.category, p.subcategory',
 'CREATE OR ALTER VIEW [bi].[vw_channel_comparison] AS\n'
 'SELECT\n'
 '  re.business_type as channel,\n'
 '  r.region_name,\n'
 '  COUNT(DISTINCT f.sales_order_number) as order_count,\n'
 '  SUM(f.quantity) as units_sold,\n'
 '  SUM(f.sales_revenue) as revenue,\n'
 '  SUM(f.sales_cost) as cost,\n'
 '  SUM(f.gross_profit) as profit\n'
 'FROM dbo.fct_sales f\n'
 'JOIN dbo.dim_reseller re ON f.reseller_sk = re.reseller_sk\n'
 'JOIN dbo.dim_region r ON f.region_sk = r.region_sk\n'
 'GROUP BY re.business_type, r.region_name']


In [ ]:
import com.microsoft.spark.fabric  # noqa: F401  registers the connector
import requests

workspace_id = spark.conf.get('trident.workspace.id')
token = mssparkutils.credentials.getToken('https://api.fabric.microsoft.com')

# Resolved at RUN time from the workspace this notebook is in, so the
# same artefact points at dev's warehouse in dev and qa's in qa.
warehouses = requests.get(
    f'https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/warehouses',
    headers={'Authorization': f'Bearer {token}'}, timeout=60).json()['value']
target = next(w for w in warehouses if w['displayName'] == WAREHOUSE)
endpoint = target['properties']['connectionString']
print(f'{WAREHOUSE} -> {endpoint}')


In [ ]:
sql_token = mssparkutils.credentials.getToken('https://database.windows.net/')
jvm = spark._jvm
props = jvm.java.util.Properties()
props.setProperty('accessToken', sql_token)
props.setProperty('encrypt', 'true')
conn = jvm.java.sql.DriverManager.getConnection(
    f'jdbc:sqlserver://{endpoint}:1433;database={WAREHOUSE}', props)
conn.setAutoCommit(True)
stmt = conn.createStatement()

# Each view is applied independently. One malformed view must not stop
# the other three -- and a single aggregate error hides how many were
# actually fine.
failed = []
for name, sql in zip(VIEWS, STATEMENTS):
    try:
        stmt.execute(sql)
        print(f'  ok      {REPORTING_SCHEMA}.{name}')
    except Exception as exc:
        failed.append(name)
        print(f'  FAILED  {REPORTING_SCHEMA}.{name}: {str(exc)[:160]}')

stmt.close(); conn.close()

if failed:
    raise RuntimeError(
        f'{len(failed)} of {len(VIEWS)} view(s) could not be created: '
        + ', '.join(failed))
print(f'\nall {len(VIEWS)} reporting view(s) are current')
